In [1]:
# Import necessary libraries
import torch
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, random_split
import os
import numpy as np
from PIL import Image
from collections import Counter
import lightning as L
from torchmetrics.classification import Accuracy
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split, LeaveOneGroupOut
from torch.utils.data import Subset
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.callbacks import ModelCheckpoint
import optuna
from optuna.integration import PyTorchLightningPruningCallback

/home/ryuko/Documents/codes/Skripsi/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# --- 1. Revised Model Architecture ---
# Based on the paper's description and diagram[cite: 189, 384].
# Key changes: Added U-Net style skip connections and adjusted decoder to match the diagram.
class ConVAT(nn.Module):
    def __init__(self, in_channels=1, num_classes=8):
        """
        Revised implementation of the ConVAT model architecture.
        This version more closely follows the U-Net-like structure with skip connections
        implied by the diagram in the paper.

        Args:
            in_channels (int): Number of input channels (1 for grayscale, 3 for color).
            num_classes (int): Number of output classes for classification.
        """
        super(ConVAT, self).__init__()

        # --- ENCODER ---
        # Extracts hierarchical features from the input image.

        # Input: 256x256
        self.enc1 = self.conv_block(in_channels, 32, 64)   # Encoder block 1
        self.pool1 = nn.MaxPool2d(2, 2)

        self.enc2 = self.conv_block(64, 128)               # Encoder block 2
        self.pool2 = nn.MaxPool2d(2, 2)

        # --- BOTTLENECK with VARIATIONAL ATTENTION ---
        # The deepest part of the network where the attention mechanism is applied.
        self.bottleneck_conv = nn.Conv2d(128, 256, kernel_size=3, padding=1) # As per diagram showing deeper features before attention
        self.relu = nn.ReLU(inplace=True)

        self.attention = nn.MultiheadAttention(embed_dim=256, num_heads=8, batch_first=True) # [cite: 185]
        self.squeeze = nn.AdaptiveAvgPool2d(1) # [cite: 233]

        # --- DECODER ---
        # Reconstructs spatial information using features from the encoder (skip connections).

        self.upconv1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(256, 128, 64) # Input channels = 128 (from upconv) + 128 (from enc2 skip)

        self.upconv2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(96, 64, 32)  # Input channels = 32 (from upconv) + 64 (from enc1 skip)

        # --- CLASSIFICATION HEAD ---
        # Fully connected layers for the final classification task.
        self.dense1 = nn.Linear(256, 1024)
        self.dropout1 = nn.Dropout(0.5)

        self.dense2 = nn.Linear(1024, 512)
        self.bn1 = nn.BatchNorm1d(1024)
        self.bn2 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(0.5)

        # self.output_dense = nn.Sequential(
        #                         nn.Linear(512, num_classes),
        #                         nn.Softmax(dim=1)
        #                     )

        self.output_dense = nn.Linear(512, num_classes)

        # --- SEGMENTATION HEAD (as shown in diagram) ---
        # Final convolution to produce a segmentation map.
        self.final_conv = nn.Conv2d(32, 1, kernel_size=1) # [cite: 328]
        self.sigmoid_out = nn.Sigmoid() # [cite: 328]


    def conv_block(self, in_channels, *filters):
        """Helper to create a sequence of Conv2D and ReLU layers."""
        layers = []
        current_channels = in_channels
        for f in filters:
            layers.append(nn.Conv2d(current_channels, f, kernel_size=3, padding=1))
            layers.append(nn.ReLU(inplace=True))
            current_channels = f
        return nn.Sequential(*layers)

    def forward(self, x):
        # --- ENCODER PATH ---
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        # --- BOTTLENECK & ATTENTION ---
        bottleneck = self.relu(self.bottleneck_conv(p2))

        batch_size, channels, height, width = bottleneck.shape
        attn_input = bottleneck.view(batch_size, channels, -1).permute(0, 2, 1) # Reshape for MHA
        attn_output, _ = self.attention(attn_input, attn_input, attn_input)
        attn_output = attn_output.permute(0, 2, 1).view(batch_size, channels, height, width) # Reshape back

        # --- CLASSIFICATION PATH ---
        squeezed = self.squeeze(attn_output)
        squeezed = squeezed.view(batch_size, -1)

        x = self.dense1(squeezed)
        x = self.relu(x)
        x = self.bn1(x)
        d1 = self.dropout1(x)

        x = self.dense2(d1)
        x = self.relu(x)
        x = self.bn2(x)
        d2 = self.dropout2(x)

        class_output = self.output_dense(d2)

        # --- DECODER PATH (with skip connections) ---
        u1 = self.upconv1(attn_output)
        # Concatenate skip connection from encoder 2
        merged1 = torch.cat([u1, e2], dim=1)
        d1_dec = self.dec1(merged1)

        u2 = self.upconv2(d1_dec)
        # Concatenate skip connection from encoder 1
        merged2 = torch.cat([u2, e1], dim=1)
        d2_dec = self.dec2(merged2)

        # The paper's architecture diagram also includes a segmentation output.
        # While the main task is classification, we can compute it to fully match the diagram.
        seg_output = self.sigmoid_out(self.final_conv(d2_dec))

        # The primary return value is for the classification task.
        return class_output

In [3]:
class MicroExpClassifier(L.LightningModule):
    def __init__(self, model: nn.Module, lr: float = 1e-3, num_classes: int = 8, weight_decay: float = 1e-3, optimizer_name: str = "Adam", label_smoothing: float = 0.1, scheduler_patience: int = 5):
        super().__init__()
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay
        self.optimizer_name = optimizer_name
        self.scheduler_patience = scheduler_patience
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.save_hyperparameters(ignore=['model'])

        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes)
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        data, targets = batch
        scores = self(data)
        loss = self.criterion(scores, targets)

        # log metric
        self.train_acc.update(scores, targets)
        self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        self.log("train_acc", self.train_acc, prog_bar=True, on_epoch=True, on_step=False)

        return loss

    def validation_step(self, batch, batch_idx):
        data, targets = batch
        scores = self(data)
        loss = self.criterion(scores, targets)

        self.val_acc.update(scores, targets)
        self.log("val_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        self.log("val_acc", self.val_acc, prog_bar=True, on_epoch=True, on_step=False)

    def test_step(self, batch, batch_idx):
        data, targets = batch
        scores = self(data)
        loss = self.criterion(scores, targets)

        self.test_acc.update(scores, targets)
        self.log("test_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        self.log("test_acc", self.test_acc, prog_bar=True, on_epoch=True, on_step=False)

    def configure_optimizers(self):
        if self.hparams.optimizer_name == "AdamW":
            optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        elif self.hparams.optimizer_name == "RMSprop":
            optimizer = torch.optim.RMSprop(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        else:
            optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.1,
            patience=self.hparams.scheduler_patience,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }


In [5]:
# --- Training Parameters ---
ROOT_DIR = "onset_apex//"

# LEARNING_RATE = 0.001
LEARNING_RATE = 0.00017259054401438885
WEIGHT_DECAY = 0.00022470394819105189
OPTIMIZER = "RMSprop"
LABEL_SMOOTHING= 0.18097191552437986
SCHEDULER_PATIENCE = 4


BATCH_SIZE = 16
# N_SPLITS = 5
EPOCHS = 15 # The paper uses different epochs for each dataset (15 for SAMM, 25 for CASME II, 35 for SMIC) [cite: 342]
NUM_CLASSES = 3 # Example, adjust based on the dataset (SAMM: 8, CASME II: 7, SMIC: 3) [cite: 173]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 8
IN_CHANNELS = 3 # Number of input channels (1 for grayscale, 3 for color)

In [6]:

# Automatically detect classes from folder names
class_names = sorted([d for d in os.listdir(ROOT_DIR) if os.path.isdir(os.path.join(ROOT_DIR, d))])
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for i, name in enumerate(class_names)}

print(f"Detected classes: {class_to_idx}")

all_subjects = []
labels_of_subjects = []

# Loop to gather each subject's folder path and its class label
for class_name in class_names:
    class_path = os.path.join(ROOT_DIR, class_name)
    for subject_id in os.listdir(class_path):
        subject_path = os.path.join(class_path, subject_id)
        if os.path.isdir(subject_path):
            all_subjects.append(subject_path)
            labels_of_subjects.append(class_to_idx[class_name])

subjects_array = np.array(all_subjects)
labels_array = np.array(labels_of_subjects)
group_ids = np.arange(len(subjects_array))
num_subjects = len(all_subjects)
print(f"Total unique subjects found: {num_subjects}")

Detected classes: {'Rendah': 0, 'Sedang': 1, 'Tinggi': 2}
Total unique subjects found: 29


In [7]:
class SubjectBasedDataset(Dataset):
    def __init__(self, subject_dirs, class_to_idx, transform=None):
        """
        Initializes the dataset from a list of subject directories.
        Args:
            subject_dirs (list): A list of paths to the subject folders.
            class_to_idx (dict): A dictionary mapping class names to indices.
            transform (callable, optional): Transformations to be applied to the images.
        """
        self.transform = transform
        self.samples = []
        self.class_to_idx = class_to_idx

        # Populate the samples list from the provided subject directories
        for subject_dir in subject_dirs:
            # Infer class name from the subject's parent directory
            class_name = os.path.basename(os.path.dirname(subject_dir))
            class_idx = self.class_to_idx[class_name]

            # Gather all images from this subject's folder
            for file_name in sorted(os.listdir(subject_dir)): # 'sorted' for consistent order
                if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(subject_dir, file_name)
                    self.samples.append((img_path, class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


In [8]:
# --- Data Augmentation and Transforms ---

train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [ ]:
# # Split subjek 60% train adn 40% validation
# train_subjects_tuning, val_subjects_tuning, _, _ = train_test_split(
#     subjects_array,
#     labels_array,
#     test_size=0.4,
#     stratify=labels_array,
#     random_state=42
# )

# # dataset for tunning
# train_dataset_tuning = SubjectBasedDataset(train_subjects_tuning.tolist(), class_to_idx, transform=train_transforms)
# valid_dataset_tuning = SubjectBasedDataset(val_subjects_tuning.tolist(), class_to_idx, transform=val_test_transforms)

# print(f"Jumlah subjek untuk tuning: {len(train_subjects_tuning)} (train), {len(val_subjects_tuning)} (validasi)")
# print(f"Jumlah gambar untuk tuning: {len(train_dataset_tuning)} (train), {len(valid_dataset_tuning)} (validasi)")

In [ ]:
# def objective(trial: optuna.trial.Trial):
#     lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
#     weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
#     optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "Adam", "RMSprop"])
#     label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.2)
#     scheduler_patience = trial.suggest_int("scheduler_patience", 3, 10)

#     model_compiled = torch.compile(ConVAT(
#         in_channels=IN_CHANNELS,
#         num_classes=NUM_CLASSES,
#     ))

#     lit_model = MicroExpClassifier(
#         model=model_compiled,
#         lr=lr,
#         optimizer_name=optimizer_name,
#         num_classes=NUM_CLASSES,
#         weight_decay=weight_decay,
#         label_smoothing=label_smoothing,
#         scheduler_patience=scheduler_patience
#     )

#     num_tuning_samples = int(len(train_dataset_tuning) * 0.50)
#     tuning_indices = np.random.choice(len(train_dataset_tuning), num_tuning_samples, replace=False)
#     train_subset_for_tuning = Subset(train_dataset_tuning, tuning_indices)

#     train_loader_tuning = DataLoader(train_subset_for_tuning, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
#     valid_loader_tuning = DataLoader(valid_dataset_tuning, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

#     pruning_callback = PyTorchLightningPruningCallback(trial, monitor="val_acc")

#     trainer = L.Trainer(
#         max_epochs=10,
#         accelerator=DEVICE,
#         devices=1,
#         precision='16-mixed',
#         accumulate_grad_batches=BATCH_SIZE,
#         callbacks=[pruning_callback],
#         logger=False,
#         enable_progress_bar=True
#     )

#     trainer.fit(lit_model, train_loader_tuning, valid_loader_tuning)

#     val_acc_value = trainer.callback_metrics.get("val_acc", torch.tensor(0.0)).item()
#     return val_acc_value

In [ ]:
# study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
# study.optimize(objective, n_trials=20)

# print("Proses tuning selesai.")
# print("Trial terbaik:")
# best_trial = study.best_trial

# print(f"  Value (Max val_acc): {best_trial.value:.4f}")
# print("  Params: ")
# for key, value in best_trial.params.items():
#     print(f"    {key}: {value}")

# best_params = best_trial.params

Trial terbaik:
  Value (Max val_acc): 0.5989
  Params: 
    lr: 0.00017259054401438885
    weight_decay: 0.00022470394819105189
    optimizer: RMSprop
    label_smoothing: 0.18097191552437986
    scheduler_patience: 4

In [ ]:
# --- LOSO Cross-Validation Loop ---
torch.set_float32_matmul_precision("medium")
logo = LeaveOneGroupOut()
test_accuracies = []

for fold, (train_val_idx, test_idx) in enumerate(logo.split(subjects_array, labels_array, groups=group_ids)):
    print(f"\n{'='*20} FOLD {fold + 1}/{num_subjects} {'='*20}")

    test_subject_id = group_ids[test_idx[0]]
    test_mask = group_ids == test_subject_id
    test_image_paths = subjects_array[test_mask]
    test_labels = labels_array[test_mask]

    print(f"Testing on Subject ID: {test_subject_id} with {len(test_image_paths)} images")

    train_val_subjects = subjects_array[train_val_idx]
    train_val_labels = labels_array[train_val_idx]

    unique_train_subjects = np.unique(train_val_subjects)

    subject_to_label = {
        subj: train_val_labels[np.where(train_val_subjects == subj)[0][0]]
        for subj in unique_train_subjects
    }

    train_subj, val_subj = train_test_split(
        unique_train_subjects,
        test_size=0.2,
        random_state=42,
        stratify=[subject_to_label[s] for s in unique_train_subjects]
    )

    train_mask = np.isin(train_val_subjects, train_subj)
    val_mask = np.isin(train_val_subjects, val_subj)

    train_subjects = train_val_subjects[train_mask]
    val_subjects = train_val_subjects[val_mask]
    train_labels = train_val_labels[train_mask]
    val_labels = train_val_labels[val_mask]

    # === Dataset dan DataLoader ===
    train_dataset = SubjectBasedDataset(train_subjects.tolist(), class_to_idx, transform=train_transforms)
    valid_dataset = SubjectBasedDataset(val_subjects.tolist(), class_to_idx, transform=val_test_transforms)
    test_dataset  = SubjectBasedDataset(test_image_paths.tolist(), class_to_idx, transform=val_test_transforms)

    train_labels_list = [label for _, label in train_dataset.samples]
    class_counts = Counter(train_labels_list)
    class_weights = 1.0 / torch.tensor([class_counts[i] for i in range(NUM_CLASSES)], dtype=torch.float)
    samples_weight = torch.tensor([class_weights[label] for label in train_labels_list])
    train_sampler = WeightedRandomSampler(weights=samples_weight, num_samples=len(samples_weight), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS, drop_last=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print(f"Train images: {len(train_dataset)}, Valid images: {len(valid_dataset)}, Test images: {len(test_dataset)}")

    # === Model Setup ===
    model_compiled = torch.compile(ConVAT(
        in_channels=IN_CHANNELS,
        num_classes=NUM_CLASSES,
    ))

    lit_model = MicroExpClassifier(
        model=model_compiled,
        lr=LEARNING_RATE,
        optimizer_name=OPTIMIZER,
        num_classes=NUM_CLASSES,
        weight_decay=WEIGHT_DECAY,
        label_smoothing=LABEL_SMOOTHING,
        scheduler_patience=SCHEDULER_PATIENCE
    )

    checkpoint_callback = ModelCheckpoint(
        monitor='val_loss',
        dirpath=f'checkpoints/loso_fold_{fold+1}/',
        filename='convat-best-{epoch:02d}-{val_loss:.2f}',
        save_top_k=1,
        mode='min',
    )
    early_stop_callback = EarlyStopping(monitor='val_loss', patience=5, mode='min')

    trainer = L.Trainer(
        max_epochs=EPOCHS,
        accelerator=DEVICE,
        devices=1,
        log_every_n_steps=5,
        callbacks=[early_stop_callback, checkpoint_callback],
        logger=TensorBoardLogger("lightning_logs", name=f"convat_fold_{fold+1}"),
        enable_progress_bar=True
    )

    trainer.fit(lit_model, train_loader, valid_loader)

    print(f"--- Testing Fold {fold + 1} on held-out subject ---")
    results = trainer.test(lit_model, dataloaders=test_loader, ckpt_path='best')
    test_accuracy = results[0]['test_acc']
    test_accuracies.append(test_accuracy)
    print(f"Accuracy for Fold {fold+1}: {test_accuracy:.4f}")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/ryuko/Documents/codes/Skripsi/.venv/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:751: Checkpoint directory /home/ryuko/Documents/codes/Skripsi/checkpoints/loso_fold_1 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | model     | OptimizedModule    | 2.0 M  | train
1 | criterion | CrossEntropyLoss   | 0      | train
2 | train_acc | MulticlassAccuracy | 0      | train
3 | val_acc   | MulticlassAccuracy | 0      | train
4 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
2.0 M     Trainable params
0         Non-trainable params
2.0 M     Total params
8.105     Total estimated model params size (MB)
44        Modules in train mode
0         Modules in 


==================== FOLD 1/29 ====================
Testing on Subject ID: 0 with 1 images
Train images: 28146, Valid images: 10983, Test images: 12
Epoch 6: 100%|██████████| 1759/1759 [04:10<00:00,  7.03it/s, v_num=18, val_loss=1.020, val_acc=0.550, train_loss=0.679, train_acc=0.994]


Restoring states from the checkpoint path at /home/ryuko/Documents/codes/Skripsi/checkpoints/loso_fold_1/convat-best-epoch=01-val_loss=0.88.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


--- Testing Fold 1 on held-out subject ---


Loaded model weights from the checkpoint at /home/ryuko/Documents/codes/Skripsi/checkpoints/loso_fold_1/convat-best-epoch=01-val_loss=0.88.ckpt


...

--- Testing Fold 3 on held-out subject ---


Loaded model weights from the checkpoint at /home/ryuko/Documents/codes/Skripsi/checkpoints/loso_fold_3/convat-best-epoch=06-val_loss=0.86.ckpt


Testing DataLoader 0: 100%|██████████| 5/5 [00:00<00:00, 24.74it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc                    0.0
        test_loss           1.4911165237426758
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Accuracy for Fold 3: 0.0000

==================== FOLD 4/29 ====================
Testing on Subject ID: 3 with 1 images
Train images: 26819, Valid images: 10983, Test images: 1339


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | model     | OptimizedModule    | 2.0 M  | train
1 | criterion | CrossEntropyLoss   | 0      | train
2 | train_acc | MulticlassAccuracy | 0      | train
3 | val_acc   | MulticlassAccuracy | 0      | train
4 | test_acc  | MulticlassAccuracy | 0      | train
---------------------------------------------------------
2.0 M     Trainable params
0         Non-trainable params
2.0 M     Total params
8.105     Total estimated model params size (MB)
44        Modules in train mode
0         Modules in eval mode


Epoch 4:  52%|█████▏    | 874/1676 [01:47<01:39,  8.10it/s, v_num=3, val_loss=0.932, val_acc=0.667, train_loss=0.685, train_acc=0.985] 

In [ ]:

# # Split subjects into training (60%) and a temporary set (40%)
# train_subjects, temp_subjects, _, _ = train_test_split(
#     all_subjects,
#     labels_of_subjects,
#     test_size=0.4,  # 40% for validation + test
#     stratify=labels_of_subjects,
#     random_state=42
# )

# # Split the temporary set into validation (20%) and test (20%)
# # Note: test_size=0.5 means 50% of the 40% temp set.
# val_subjects, test_subjects, _, _ = train_test_split(
#     temp_subjects,
#     [labels_of_subjects[all_subjects.index(s)] for s in temp_subjects],
#     test_size=0.5,
#     stratify=[labels_of_subjects[all_subjects.index(s)] for s in temp_subjects],
#     random_state=42
# )

# print(f"\nNumber of training subjects: {len(train_subjects)}")
# print(f"Number of validation subjects: {len(val_subjects)}")
# print(f"Number of testing subjects: {len(test_subjects)}")

In [ ]:
# train_dataset = SubjectBasedDataset(train_subjects, class_to_idx, transform=train_transforms)
# valid_dataset = SubjectBasedDataset(val_subjects, class_to_idx, transform=val_test_transforms)
# test_dataset  = SubjectBasedDataset(test_subjects, class_to_idx, transform=val_test_transforms)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
# valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
# test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# print(f"\nNumber of training images: {len(train_dataset)}")
# print(f"Number of validation images: {len(valid_dataset)}")
# print(f"Number of testing images: {len(test_dataset)}")

In [ ]:
# train_labels = [label for _, label in train_dataset.samples]
# class_counts = Counter(train_labels)
# print(f"Class distribution in training set: {sorted(class_counts.items())}")

# # Ensure NUM_CLASSES is defined (e.g., NUM_CLASSES = len(class_names))
# class_weights = 1.0 / torch.tensor([class_counts[i] for i in range(NUM_CLASSES)], dtype=torch.float)

# samples_weight = torch.tensor([class_weights[label] for label in train_labels])

# train_sampler = WeightedRandomSampler(
#     weights=samples_weight,
#     num_samples=len(samples_weight),
#     replacement=True
# )

In [ ]:
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS, drop_last=True)
# valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=True)
# test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, drop_last=True)

In [ ]:
# from collections import Counter
# import matplotlib.pyplot as plt

# # Hitung distribusi label di training set
# class_counts = Counter(train_labels)
# print("Distribusi label di training set:", class_counts)

# # Kalau mau persentase
# total = sum(class_counts.values())
# for cls, count in class_counts.items():
#     print(f"Class {cls}: {count} ({count/total:.2%})")

# # Visualisasi distribusi label
# plt.bar(class_counts.keys(), class_counts.values())
# plt.xlabel("Class")
# plt.ylabel("Count")
# plt.title("Distribusi Label Training Set")
# plt.show()


In [ ]:
# # Lightning model
# lit_model = MicroExpClassifier(model=model, lr=LEARNING_RATE, num_classes=NUM_CLASSES)

# torch.set_float32_matmul_precision("medium")

# logger = TensorBoardLogger("lightning_logs", name="convat_samm")
# early_stop_callback = EarlyStopping(
#     monitor='val_loss',
#     patience=15,
#     verbose=True,
#     mode='min'
# )

# # Checkpointing
# checkpoint_callback = ModelCheckpoint(
#     monitor='val_loss',
#     dirpath='checkpoints/data_polinema/',
#     filename='convat-best-{epoch:02d}-{val_loss:.2f}',
#     save_top_k=1,
#     mode='min',
#     verbose=True
# )

# # trainer
# trainer = L.Trainer(
#     max_epochs=EPOCHS,
#     accelerator="gpu" if torch.cuda.is_available() else "cpu",
#     devices=1,
#     log_every_n_steps=10,
#     callbacks=[early_stop_callback, checkpoint_callback],
#     logger=logger,
# )

# # Run
# trainer.fit(lit_model, train_loader, valid_loader)

In [ ]:
trainer.test(lit_model, dataloaders=test_loader)

trainer.save_checkpoint("checkpoints/data_polinema/convat-1-last.ckpt")

In [ ]:
BEST_MODEL_PATH = "/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/checkpoints/data_polinema/convat-best-epoch=03-val_loss=0.03.ckpt"

# Muat model dari checkpoint
best_model = MicroExpClassifier.load_from_checkpoint(BEST_MODEL_PATH, model=model)

In [ ]:
best_model.eval()

# Buat trainer baru hanya untuk testing
tester = L.Trainer(accelerator='gpu', devices=1)

# Jalankan evaluasi pada test_loader
print("========================================")
print("Running Final Evaluation on the Test Set...")
print("========================================")
tester.test(model=best_model, dataloaders=test_loader)

In [ ]:
def evaluate_model(model, dataloader, device):
    model = model.to(device)
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            preds = scores.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = correct / total
    cm = confusion_matrix(all_labels, all_preds)

    print(f"Validation Accuracy: {acc:.4f}")
    print("Confusion Matrix:")
    print(cm)

    return acc, cm

In [ ]:
acc, cm = evaluate_model(lit_model, valid_loader, DEVICE)